In [1]:
# Import libraries and load chart events safely (chartevents can be very large).
from pathlib import Path

import pandas as pd

data_path = Path("../data/mimic-iv-3.1/icu/chartevents.csv.gz")

# Keep only columns needed for feature engineering.
usecols = ["subject_id", "hadm_id", "stay_id", "charttime", "itemid", "valuenum"]
dtype_map = {
    "subject_id": "int32",
    "hadm_id": "int32",
    "stay_id": "int32",
    "itemid": "int32",
    "valuenum": "float32",
}

# Restrict to key vital-sign itemids to reduce memory footprint.
target_itemids = {220045, 220179, 220180, 220181, 220210, 223761, 220277}

# Optional random cap for notebook experimentation. Set to None for full filtered data.
max_rows = 1_000_000

chunks = []
for chunk in pd.read_csv(
    data_path,
    compression="gzip",
    usecols=usecols,
    dtype=dtype_map,
    parse_dates=["charttime"],
    chunksize=500_000,
    low_memory=True,
):
    chunk = chunk[chunk["valuenum"].notna()]
    chunk = chunk[chunk["itemid"].isin(target_itemids)]
    chunks.append(chunk)

chartevents = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame(columns=usecols)

# Sample after loading filtered rows to avoid file-order truncation bias.
if max_rows is not None and len(chartevents) > max_rows:
    chartevents = chartevents.sample(n=max_rows, random_state=42).reset_index(drop=True)

print(f"Loaded {len(chartevents):,} rows after filtering")
chartevents.head()

Loaded 1,000,000 rows after filtering


,subject_id,hadm_id,stay_id,charttime,itemid,valuenum
0,11389314,25398830,37114814,2193-12-14 18:01:00,220179,95.0
1,13261557,24351231,36761888,2166-05-29 07:45:00,220045,67.0
2,11552741,24281438,35839750,2160-01-30 21:00:00,220045,86.0
3,12805811,23007640,30056217,2170-03-07 02:00:00,220210,16.0
4,12952402,22152100,34933034,2158-12-07 05:14:00,220179,165.0


In [2]:
# Select important vital signs
vital_itemids = [
    220045,  # Heart Rate
    220179,  # Systolic Blood Pressure
    220180,  # Diastolic Blood Pressure
    220181,  # Mean Blood Pressure
    220210,  # Respiratory Rate
    223761,  # Temperature
    220277,  # Oxygen Saturation
]

# Filter chartevents for selected vital signs only.
vitals = chartevents[chartevents["itemid"].isin(vital_itemids)].copy()
vitals.head()

,subject_id,hadm_id,stay_id,charttime,itemid,valuenum
0,11389314,25398830,37114814,2193-12-14 18:01:00,220179,95.0
1,13261557,24351231,36761888,2166-05-29 07:45:00,220045,67.0
2,11552741,24281438,35839750,2160-01-30 21:00:00,220045,86.0
3,12805811,23007640,30056217,2170-03-07 02:00:00,220210,16.0
4,12952402,22152100,34933034,2158-12-07 05:14:00,220179,165.0


In [3]:
#keep important columns
vitals = vitals[["subject_id", "hadm_id", "stay_id", "itemid", "valuenum"]]

In [4]:
# Aggregate vital signs by ICU stay
vitals_agg = vitals.groupby(["stay_id", "itemid"])["valuenum"].mean().unstack()
vitals_agg.head()

itemid,220045,220179,220180,220181,220210,220277,223761
stay_id,,,,,,,
30000153,100.000000,NaN,NaN,82.000000,NaN,NaN,99.050003
30000213,87.000000,151.0,56.5,NaN,27.0,90.000000,98.900002
30000484,NaN,111.0,NaN,75.000000,NaN,NaN,NaN
30000646,90.500000,89.5,59.0,63.799999,20.5,94.666664,98.400002
30000831,112.599998,123.0,62.5,87.000000,25.5,93.333336,NaN


In [5]:
# Rename columns for readability
vitals_agg = vitals_agg.rename(columns={
    220045: "heart_rate",
    220179: "sbp",
    220180: "dbp",
    220181: "mbp",
    220210: "resp_rate",
    223761: "temperature",
    220277: "spo2",
})

vitals_agg.head()

itemid,heart_rate,sbp,dbp,mbp,resp_rate,spo2,temperature
stay_id,,,,,,,
30000153,100.000000,NaN,NaN,82.000000,NaN,NaN,99.050003
30000213,87.000000,151.0,56.5,NaN,27.0,90.000000,98.900002
30000484,NaN,111.0,NaN,75.000000,NaN,NaN,NaN
30000646,90.500000,89.5,59.0,63.799999,20.5,94.666664,98.400002
30000831,112.599998,123.0,62.5,87.000000,25.5,93.333336,NaN


In [6]:
#making sure each ICU stay appears only once in the dataset
vitals_agg.index.nunique()
vitals_agg.shape[0]

90588

In [7]:
# merge vitals into main dataset (safe to rerun)
vitals_agg_reset = vitals_agg.reset_index()

# If a base dataframe is not loaded yet, build one ICU-stay table from chartevents.
if "df" not in globals():
    df = chartevents[["subject_id", "hadm_id", "stay_id"]].drop_duplicates("stay_id")

# Remove old vital columns before merging again to avoid duplicate-column errors.
vital_cols = [c for c in vitals_agg_reset.columns if c != "stay_id"]
df = df.drop(columns=[c for c in vital_cols if c in df.columns], errors="ignore")

df = df.merge(vitals_agg_reset, on="stay_id", how="left")
df.head()
df.describe()


,subject_id,hadm_id,stay_id,heart_rate,sbp,dbp,mbp,resp_rate,spo2,temperature
count,9.058800e+04,9.058800e+04,9.058800e+04,63273.000000,51269.000000,50953.000000,51059.000000,62556.000000,6.241600e+04,28470.000000
mean,1.500591e+07,2.498487e+07,3.499624e+07,85.947823,119.949310,68.222664,79.594933,19.398870,1.680362e+02,98.517578
std,2.885641e+06,2.884015e+06,2.886275e+06,302.798004,53.759998,348.515594,28.261478,4.914512,1.781179e+04,9.623932
min,1.000003e+07,2.000009e+07,3.000015e+07,0.000000,0.000000,0.000000,12.000000,0.000000,0.000000e+00,0.000000
25%,1.251427e+07,2.248595e+07,3.250384e+07,73.000000,105.000000,56.000000,69.500000,16.000000,9.500000e+01,97.900002
50%,1.500610e+07,2.498768e+07,3.499843e+07,83.750000,118.000000,65.000000,78.000000,19.000000,9.700000e+01,98.399994
75%,1.752034e+07,2.746916e+07,3.748841e+07,95.000000,132.000000,74.500000,88.000000,22.000000,9.845454e+01,98.900002
max,1.999999e+07,2.999983e+07,3.999986e+07,76137.500000,10211.700195,75105.000000,5546.000000,202.750000,4.450046e+06,986.000000


In [8]:
# Add age, gender, LOS, and mortality from ICU/HOSP tables
from pathlib import Path

base_path = Path("../data/mimic-iv-3.1")

icustays = pd.read_csv(
    base_path / "icu" / "icustays.csv.gz",
    compression="gzip",
    usecols=["subject_id", "hadm_id", "stay_id", "los"],
)

admissions = pd.read_csv(
    base_path / "hosp" / "admissions.csv.gz",
    compression="gzip",
    usecols=["hadm_id", "hospital_expire_flag"],
)

patients = pd.read_csv(
    base_path / "hosp" / "patients.csv.gz",
    compression="gzip",
    usecols=["subject_id", "gender", "anchor_age"],
)

demo = (
    icustays
    .merge(admissions, on="hadm_id", how="left", validate="many_to_one")
    .merge(patients, on="subject_id", how="left", validate="many_to_one")
)

demo["age"] = demo["anchor_age"]
demo["gender"] = demo["gender"].map({"M": 0, "F": 1})
demo["mortality"] = demo["hospital_expire_flag"].astype("float")

demo = demo[["stay_id", "age", "gender", "los", "mortality"]].drop_duplicates("stay_id")

# If rerun, drop old demo columns first so merge remains idempotent.
demo_cols = ["age", "gender", "los", "mortality"]
df = df.drop(columns=[c for c in demo_cols if c in df.columns], errors="ignore")
df = df.merge(demo, on="stay_id", how="left")

expected_cols = [
    "age", "gender", "los",
    "heart_rate", "sbp", "dbp", "mbp", "resp_rate", "spo2", "temperature",
    "mortality",
]

df[expected_cols].head()

,age,gender,los,heart_rate,sbp,dbp,mbp,resp_rate,spo2,temperature,mortality
0,53,1,2.190255,79.000000,111.000000,84.000000,108.000000,NaN,97.75,NaN,0.0
1,66,0,4.948796,66.666664,100.599998,53.200001,60.000000,15.500000,95.00,98.400002,0.0
2,84,0,2.518241,78.500000,NaN,56.000000,80.333336,17.000000,90.00,NaN,0.0
3,55,0,10.411493,87.750000,NaN,NaN,35.000000,18.333334,99.00,97.649994,1.0
4,62,1,21.485451,81.500000,151.000000,61.900002,87.750000,17.125000,98.00,99.133331,0.0


In [9]:
#checking for missing values 
df.isnull().sum()

subject_id         0
hadm_id            0
stay_id            0
heart_rate     27315
sbp            39319
dbp            39635
mbp            39529
resp_rate      28032
spo2           28172
temperature    62118
age                0
gender             0
los               14
mortality          0
dtype: int64

In [10]:
# Fill missing values using column-specific strategy (avoid target corruption)
vital_cols = ["heart_rate", "sbp", "dbp", "mbp", "resp_rate", "spo2", "temperature"]
cont_demo_cols = ["age", "los"]

for col in vital_cols + cont_demo_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

if "gender" in df.columns and df["gender"].isna().any():
    df["gender"] = df["gender"].fillna(df["gender"].mode().iloc[0])

# Mortality is the target; drop rows where it is missing, then cast to int.
df = df.dropna(subset=["mortality"]).copy()
df["mortality"] = df["mortality"].astype(int)

# Final safety checks for binary fields.
assert set(df["mortality"].unique()).issubset({0, 1}), "mortality must be binary {0,1}"
assert set(df["gender"].dropna().unique()).issubset({0, 1}), "gender must be binary {0,1}"

In [11]:
#after filling lets check the data again
df.isnull().sum()

subject_id     0
hadm_id        0
stay_id        0
heart_rate     0
sbp            0
dbp            0
mbp            0
resp_rate      0
spo2           0
temperature    0
age            0
gender         0
los            0
mortality      0
dtype: int64

In [12]:
# Define retrospective and leakage-safer feature sets
feature_cols_retro = [
    "age", "gender", "los",
    "heart_rate", "sbp", "dbp", "mbp", "resp_rate", "spo2", "temperature",
]
feature_cols_safe = [
    "age", "gender",
    "heart_rate", "sbp", "dbp", "mbp", "resp_rate", "spo2", "temperature",
]

X_retro = df[feature_cols_retro].copy()
X_safe = df[feature_cols_safe].copy()
y = df["mortality"].copy()

print("Retro features shape:", X_retro.shape)
print("Safe features shape:", X_safe.shape)

Retro features shape: (90588, 10)
Safe features shape: (90588, 9)


In [13]:
from pathlib import Path
import json

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Default to leakage-safer export for downstream modeling.
handoff_view = "safe"  # choose from: "safe", "retro"

if handoff_view == "safe":
    export_cols = feature_cols_safe + ["mortality"]
    output_path = output_dir / "icu_features_safe.csv"
elif handoff_view == "retro":
    export_cols = feature_cols_retro + ["mortality"]
    output_path = output_dir / "icu_features_retro.csv"
else:
    raise ValueError("handoff_view must be either 'safe' or 'retro'.")

export_df = df[export_cols].copy()
export_df.to_csv(output_path, index=False)

meta_path = output_dir / f"{output_path.stem}_meta.json"
meta = {
    "handoff_view": handoff_view,
    "features": export_cols[:-1],
    "target": "mortality",
    "rows": int(export_df.shape[0]),
    "columns": export_df.columns.tolist(),
}
meta_path.write_text(json.dumps(meta, indent=2))

print(f"Saved features to: {output_path.resolve()}")
print(f"Saved metadata to: {meta_path.resolve()}")
display(export_df.head())

Saved features to: /Users/ramkumawat/Desktop/Projects /ICU-Mortality-Prediction/data/processed/icu_features_safe.csv
Saved metadata to: /Users/ramkumawat/Desktop/Projects /ICU-Mortality-Prediction/data/processed/icu_features_safe_meta.json


,age,gender,heart_rate,sbp,dbp,mbp,resp_rate,spo2,temperature,mortality
0,53,1,79.000000,111.000000,84.000000,108.000000,19.000000,97.75,98.399994,0
1,66,0,66.666664,100.599998,53.200001,60.000000,15.500000,95.00,98.400002,0
2,84,0,78.500000,118.000000,56.000000,80.333336,17.000000,90.00,98.399994,0
3,55,0,87.750000,118.000000,65.000000,35.000000,18.333334,99.00,97.649994,1
4,62,1,81.500000,151.000000,61.900002,87.750000,17.125000,98.00,99.133331,0
